In [1]:
raw_path = "../data/raw/historical_data.csv"

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [3]:
from src.data_modeling.data_cleaning import load_csv, parse_timestamps, clean_data
from src.data_modeling.feature_engineering import split_chronologically, create_date_features
from src.data_modeling.preprocessing import preprocessor


In [4]:
df = load_csv(file_path=raw_path)
df = parse_timestamps(df, date_cols=["created_at", "actual_delivery_time"])
df = clean_data(df)

df.to_csv("../data/clean/data_cleaned.csv")
df.info()

<class 'pandas.DataFrame'>
Index: 197421 entries, 0 to 197427
Data columns (total 17 columns):
 #   Column                                        Non-Null Count   Dtype              
---  ------                                        --------------   -----              
 0   market_id                                     197421 non-null  str                
 1   created_at                                    197421 non-null  datetime64[us, UTC]
 2   actual_delivery_time                          197421 non-null  datetime64[us, UTC]
 3   store_id                                      197421 non-null  int64              
 4   store_primary_category                        197421 non-null  str                
 5   order_protocol                                197421 non-null  str                
 6   total_items                                   197421 non-null  int64              
 7   subtotal                                      197421 non-null  int64              
 8   num_distinct_items  

In [5]:
df.columns

Index(['market_id', 'created_at', 'actual_delivery_time', 'store_id',
       'store_primary_category', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'delivery_duration'],
      dtype='str')

In [6]:
train_df, test_df = split_chronologically(df)

train_df = create_date_features(train_df)
test_df = create_date_features(test_df)

train_df.columns

Index(['market_id', 'created_at', 'actual_delivery_time', 'store_id',
       'store_primary_category', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'delivery_duration',
       'month', 'day_of_month', 'day_of_week', 'hour'],
      dtype='str')

In [7]:
target_col = "delivery_duration"

X_train = train_df.drop(
    columns=[
        target_col,
        "actual_delivery_time",
        "store_id",
        "created_at",
        "store_primary_category",
    ]
)

y_train = train_df[target_col]

X_test = test_df.drop(
    columns=[
        target_col,
        "actual_delivery_time",
        "store_id",
        "created_at",
        "store_primary_category",
    ]
)

y_test = test_df[target_col]

# Fit preprocessing ONLY on training data
X_train_processed = preprocessor.fit_transform(X_train)
# Apply the fitted preprocessing to test data
X_test_processed = preprocessor.transform(X_test)

In [14]:
import numpy as np

np.save("../data/preprocessed/X_train_processed.npy", X_train_processed)
np.save("../data/preprocessed/X_test_processed.npy", X_test_processed)

np.save("../data/preprocessed/y_train.npy", y_train.to_numpy())
np.save("../data/preprocessed/y_test.npy", y_test.to_numpy())

In [8]:
X_train.columns

Index(['market_id', 'order_protocol', 'total_items', 'subtotal',
       'num_distinct_items', 'min_item_price', 'max_item_price',
       'total_onshift_dashers', 'total_busy_dashers',
       'total_outstanding_orders', 'estimated_order_place_duration',
       'estimated_store_to_consumer_driving_duration', 'month', 'day_of_month',
       'day_of_week', 'hour'],
      dtype='str')

In [ ]:
X_train_processed.shape 

(157936, 34)

In [12]:
feature_names = preprocessor.get_feature_names_out()

print(len(feature_names))
print(feature_names)


34
['categorical__order_protocol_1.0' 'categorical__order_protocol_2.0'
 'categorical__order_protocol_3.0' 'categorical__order_protocol_4.0'
 'categorical__order_protocol_5.0' 'categorical__order_protocol_6.0'
 'categorical__order_protocol_7.0' 'categorical__order_protocol_unknown'
 'categorical__market_id_1.0' 'categorical__market_id_2.0'
 'categorical__market_id_3.0' 'categorical__market_id_4.0'
 'categorical__market_id_5.0' 'categorical__market_id_6.0'
 'categorical__market_id_unknown' 'numerical__total_items'
 'numerical__subtotal' 'numerical__num_distinct_items'
 'numerical__min_item_price' 'numerical__max_item_price'
 'numerical__total_onshift_dashers' 'numerical__total_busy_dashers'
 'numerical__total_outstanding_orders'
 'numerical__estimated_order_place_duration'
 'numerical__estimated_store_to_consumer_driving_duration'
 'numerical__month' 'numerical__day_of_month' 'numerical__day_of_week'
 'numerical__hour' 'numerical__missingindicator_min_item_price'
 'numerical__missingind